# Bölüm klipleri: ücretsiz ekran kartında hareketli video

Bu not defteri başlangıç karelerini **LTX-Video** modeliyle hareketli kliplere çevirir.
Kareler ya `npm run kaggle:paket -- --kareler` zip'inden ya da `kare-uret` not defterinin çıktısından gelir.

**Ayarlar (sağdaki panel → Session options):**
- Accelerator: **GPU T4 x2** (ya da GPU P100)
- Internet: **On** (telefon doğrulaması gerekir)
- Sağ üstte **Add Input** → kare zip'ini (dataset) ya da `kare-uret` not defterinin çıktısını ekleyin.

Sonra üstteki **Run All** düğmesine basın. Bitince sağdaki **Output** bölümünden
`klipler.zip` dosyasını indirin.

Bir oturum yarıda kalırsa tekrar **Run All** deyin: biten klipler `/kaggle/working/clips`
içinde durduğu sürece tekrar üretilmez.

In [ ]:
# 1) Kurulum (yaklaşık 2-3 dakika)
# Kaggle'ın hazır torchao sürümü güncel diffusers ile çakışıyor (FqnToConfig hatası)
!pip uninstall -y -q torchao
!pip install -q -U "diffusers>=0.34" "transformers>=4.48" accelerate sentencepiece imageio imageio-ffmpeg

In [ ]:
# 2) Ayarlar
import glob, json, os, time

# Kare zip'i Kaggle'da /kaggle/input/<dataset-adı>/ altına açılır; plan.json'u kendimiz buluyoruz.
plans = glob.glob("/kaggle/input/**/plan.json", recursive=True)
assert plans, "plan.json bulunamadı. Sağ üstte 'Add Input' ile kare zip'ini (dataset) ekleyin."
PLAN_PATH = plans[0]
plan = json.load(open(PLAN_PATH))
# Karelerin olduğu klasör (kare-uret çıktısı ayrı bir girdi olabilir)
firsts = glob.glob(f"/kaggle/input/**/keyframes/{plan['clips'][0]['key']}.png", recursive=True)
assert firsts, "Kareler bulunamadı. Önce kare-uret not defterini çalıştırıp çıktısını girdi olarak ekleyin."
DATA_DIR = os.path.dirname(os.path.dirname(firsts[0]))

OUT_DIR = "/kaggle/working/clips"
os.makedirs(OUT_DIR, exist_ok=True)

# 2B model: 13B'lik 0.9.7 sürümleri 15 GB'lık T4'e ve Kaggle'ın 29 GB RAM'ine sığmıyor.
MODEL_ID = "Lightricks/LTX-Video-0.9.5"
WIDTH, HEIGHT = 704, 416        # 32'nin katı olmalı; 16:9'a yakın, T4 belleğine rahat sığar
FPS = 24
SEG_FRAMES = 161                # bir parça ~6,7 sn (8k+1)
OVERLAP = 9                     # sonraki parça öncekinin son 9 karesinden devam eder
STEPS = 30
GUIDANCE = 3.5
SEED = 23

# Yalnızca bazı sahneleri üretmek için: ONLY = ["c03", "c04"]
ONLY = None

print(f"{plan['series']} · Bölüm {plan['episode']} · {len(plan['clips'])} sahne")
print("Kareler:", DATA_DIR)

In [ ]:
# 3) Modeli yükle (ilk seferde ~10 GB indirir)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
from diffusers import LTXConditionPipeline
from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
from diffusers.utils import export_to_video, load_image

gpu = torch.cuda.get_device_name(0)
# T4 ve P100 bfloat16'yı donanımda desteklemez (is_bf16_supported yine de True diyebilir);
# Ampere (8.x) öncesi kartlarda float16 kullanıyoruz.
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print("Ekran kartı:", gpu, "·", DTYPE)

if "pipe" not in globals():  # tekrar çalıştırmada bellek dolmasın
    pipe = LTXConditionPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
    pipe.enable_model_cpu_offload()   # metin kodlayıcı + model aynı anda kartta durmasın
    pipe.vae.enable_tiling()

In [ ]:
# 4) Klipleri üret: yavaşlatma yok; sahne uzunsa yeni parça öncekinin son karelerinden devam eder
NEGATIVE = "worst quality, static, frozen, still image, slideshow, inconsistent motion, blurry, jittery, distorted, deformed face, extra limbs, watermark, text"

done, failed = [], []
for c in plan["clips"]:
    key = c["key"]
    if ONLY and key not in ONLY:
        continue
    out = os.path.join(OUT_DIR, f"{key}.mp4")
    if os.path.exists(out):
        print(f"✔ {key} zaten var, atlandı")
        continue
    t0 = time.time()
    try:
        image = load_image(os.path.join(DATA_DIR, "keyframes", f"{key}.png")).resize((WIDTH, HEIGHT))
        # LTX uzun, hareketi anlatan komutlarla daha çok hareket eder: önce hareket, sonra sahne ve stil
        prompt = f"{c['motion']}, natural fluid movement, dynamic cinematic camera motion. {c['keyframe']}. {plan['style']}"
        need = round(c["duration"] * FPS)
        frames, seg = [], 0
        while len(frames) < need:
            cond = (LTXVideoCondition(image=image, frame_index=0) if not frames
                    else LTXVideoCondition(video=frames[-OVERLAP:], frame_index=0))
            part = pipe(
                conditions=[cond],
                prompt=prompt,
                negative_prompt=NEGATIVE,
                width=WIDTH,
                height=HEIGHT,
                num_frames=SEG_FRAMES,
                num_inference_steps=STEPS,
                guidance_scale=GUIDANCE,
                decode_timestep=0.05,
                decode_noise_scale=0.025,
                image_cond_noise_scale=0.0,
                generator=torch.Generator().manual_seed(SEED + seg),
                output_type="pil",
            ).frames[0]
            frames += part if not frames else part[OVERLAP:]
            seg += 1
            torch.cuda.empty_cache()
        export_to_video(frames[:need], out, fps=FPS)
        done.append(key)
        print(f"✔ {key}  {seg} parça · {need / FPS:.1f} sn · {time.time() - t0:.0f} sn")
    except Exception as e:
        failed.append(key)
        print(f"✖ {key}: {e}")
    torch.cuda.empty_cache()

print(f"\nBiten: {len(done)} · Hatalı: {failed or 'yok'}")

In [ ]:
# 5) Klipleri zip'le (her klip zaten sahne süresinde, gerçek hızda)
import shutil

zip_base = f"/kaggle/working/klipler-{plan['series']}-e{plan['episode']}"
shutil.make_archive(zip_base, "zip", OUT_DIR)
print("\n🎬 İndirilecek dosya:", zip_base + ".zip")
print("Sağdaki Output bölümünden indirin ve bilgisayarınızda 'npm run kaggle:al' ile yerleştirin.")